In [1]:
import os
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO = "mesale/gpt2-from-scratch"
!git clone https://{token}@github.com/{REPO}.git /kaggle/temp/repo
%cd /kaggle/temp/repo
!git remote set-url origin https://github.com/{REPO}.git
!pip install -q tokenizers pyyaml

Cloning into '/kaggle/temp/repo'...
remote: Enumerating objects: 156, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 156 (delta 58), reused 127 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (156/156), 377.19 KiB | 7.25 MiB/s, done.
Resolving deltas: 100% (58/58), done.
/kaggle/temp/repo


In [4]:
import glob
bins = glob.glob("/kaggle/input/**/tinystories_train.bin", recursive=True)
assert bins, "tinystories_train.bin not found: is the dataset attached, and did data-prep finish?"
DATA_DIR = os.path.dirname(bins[0])
print(DATA_DIR, os.listdir(DATA_DIR))
n = os.path.getsize(bins[0]) // 2          # uint16 = 2 bytes per token
print(f"{n:,} train tokens | the run sees {8000 * 32768:,} tokens = {8000 * 32768 / n:.2f} epochs")

/kaggle/input/datasets/mesaletsegaye/tinystories-v2-bpe4096/data ['shk_train.bin', 'shk_val.bin', 'tinystories_train.bin', 'tinystories_val.bin']
552,529,554 train tokens | the run sees 262,144,000 tokens = 0.47 epochs


In [5]:
# 1) controlled comparison: identical settings to the pre-LN baseline, only the placement changes
!python -m src.train --config configs/ablation_heads.yaml --set model.pre_ln=false train.run_name=postln_lr1e-3 train.data_dir={DATA_DIR} train.out_dir=/kaggle/working/runs
# 2) post-LN with a gentler schedule (the original paper needed a long warmup)
!python -m src.train --config configs/ablation_heads.yaml --set model.pre_ln=false train.lr=0.0003 train.min_lr=0.00003 train.warmup_steps=300 train.run_name=postln_lr3e-4 train.data_dir={DATA_DIR} train.out_dir=/kaggle/working/runs
# 3) optional but makes the comparison clean: pre-LN under the same gentler schedule
!python -m src.train --config configs/ablation_heads.yaml --set train.lr=0.0003 train.min_lr=0.00003 train.warmup_steps=300 train.run_name=preln_lr3e-4 train.data_dir={DATA_DIR} train.out_dir=/kaggle/working/runs

12,317,952 params | 32,768 tokens/step | device=cuda amp=True
[eval] step 0: train 8.4557 | val 8.4561
step     50 | loss 6.8274 | lr 5.00e-04 | gnorm 0.23 | 10,470 tok/s
step    100 | loss 5.9001 | lr 1.00e-03 | gnorm 0.20 | 17,116 tok/s
step    150 | loss 5.8965 | lr 9.97e-04 | gnorm 0.24 | 23,674 tok/s
step    200 | loss 5.8956 | lr 9.89e-04 | gnorm 0.33 | 31,747 tok/s
step    250 | loss 5.9051 | lr 9.75e-04 | gnorm 0.24 | 39,537 tok/s
[eval] step 250: train 5.8889 | val 5.8983
step    300 | loss 5.8894 | lr 9.56e-04 | gnorm 0.17 | 44,248 tok/s
step    350 | loss 5.8906 | lr 9.32e-04 | gnorm 0.15 | 49,985 tok/s
step    400 | loss 5.8919 | lr 9.02e-04 | gnorm 0.17 | 53,121 tok/s
step    450 | loss 5.8936 | lr 8.69e-04 | gnorm 0.17 | 56,915 tok/s
step    500 | loss 5.8926 | lr 8.31e-04 | gnorm 0.18 | 58,650 tok/s
[eval] step 500: train 5.8845 | val 5.8935
step    550 | loss 5.8903 | lr 7.90e-04 | gnorm 0.16 | 58,463 tok/s
step    600 | loss 5.8878 | lr 7.46e-04 | gnorm 0.18 | 64,194 t

In [6]:
import os, shutil, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

new_root, base = "/kaggle/working/runs", "/kaggle/temp/repo/results/ablation_heads"
res = "/kaggle/temp/repo/results/ablation_ln"
os.makedirs(res, exist_ok=True)
runs = {"pre-LN lr1e-3 (seed 1337)": f"{base}/heads_6",
        "pre-LN lr1e-3 (seed 1338)": f"{base}/heads_6_seed2"}
for n in ["postln_lr1e-3", "postln_lr3e-4", "preln_lr3e-4"]:
    if os.path.isdir(f"{new_root}/{n}"):
        runs[n] = f"{new_root}/{n}"

fig, ax = plt.subplots(figsize=(7, 4)); rows = []
for label, d in runs.items():
    ev = pd.read_csv(f"{d}/eval_log.csv")
    ax.plot(ev.step[ev.step >= 250], ev.val_loss[ev.step >= 250], "o-", label=label)
    rows.append({"run": label, "final_val": ev.val_loss.iloc[-1], "best_val": ev.val_loss.min()})
    if d.startswith(new_root):
        os.makedirs(f"{res}/{os.path.basename(d)}", exist_ok=True)
        for f in ["train_log.csv", "eval_log.csv", "config.yaml"]:
            shutil.copy(f"{d}/{f}", f"{res}/{os.path.basename(d)}")
ax.set_xlabel("step"); ax.set_ylabel("val loss (nats/token)"); ax.set_ylim(top=3); ax.legend(); ax.grid(alpha=0.3)
fig.savefig(f"{res}/val_pre_vs_post_ln.png", dpi=150, bbox_inches="tight")
table = pd.DataFrame(rows); table.to_csv(f"{res}/summary.csv", index=False)
print(table.to_string(index=False))

                      run  final_val  best_val
pre-LN lr1e-3 (seed 1337)   1.702770  1.702770
pre-LN lr1e-3 (seed 1338)   1.698366  1.698366
            postln_lr1e-3   5.888070  5.887936
            postln_lr3e-4   2.458308  2.458308
             preln_lr3e-4   1.973289  1.973289


In [7]:
%cd /kaggle/temp/repo
!git config user.email "mesalemtsegaye @gmail.com" && git config user.name "kaggle"
!git add results && git commit -m "Add pre-LN vs post-LN results" && git push https://{token}@github.com/{REPO}.git HEAD:main

/kaggle/temp/repo
[main 6260284] Add pre-LN vs post-LN results
 11 files changed, 219 insertions(+)
 create mode 100644 results/ablation_ln/postln_lr1e-3/config.yaml
 create mode 100644 results/ablation_ln/postln_lr1e-3/eval_log.csv
 create mode 100644 results/ablation_ln/postln_lr1e-3/train_log.csv
 create mode 100644 results/ablation_ln/postln_lr3e-4/config.yaml
 create mode 100644 results/ablation_ln/postln_lr3e-4/eval_log.csv
 create mode 100644 results/ablation_ln/postln_lr3e-4/train_log.csv
 create mode 100644 results/ablation_ln/preln_lr3e-4/config.yaml
 create mode 100644 results/ablation_ln/preln_lr3e-4/eval_log.csv
 create mode 100644 results/ablation_ln/preln_lr3e-4/train_log.csv
 create mode 100644 results/ablation_ln/summary.csv
 create mode 100644 results/ablation_ln/val_pre_vs_post_ln.png
Enumerating objects: 20, done.
Counting objects: 100% (20/20), done.
Delta compression using up to 4 threads
Compressing objects: 100% (18/18), done.
Writing objects: 100% (18/18), 74.7